## Initialization

In [1]:
import importlib

In [2]:
import tex # comment this if Python in your system is not "linked" with TeX.
import func_analysis
import func_input

import theoreticalmodels
importlib.reload(theoreticalmodels)
import Names
importlib.reload(Names)
import inputdata
importlib.reload(inputdata)
import clusterparams
importlib.reload(clusterparams)

# following must be loaded as they are used ahead. 
# I have commented the how to use, for example, 'colors' list from Names.py.

models = theoreticalmodels.initialize(func_input.ReadingModels)
# print(models['obisoY0248_eta02'])
names = Names.__init__()
# print(names['colors'])
dfclusters_input = inputdata.initialize(func_input.load_all_clusters_fast)
dfclusters = dict(sorted(dfclusters_input['dfclusters'].items()))
# print(dfclusters['ngc2808'])
dfparams = clusterparams.initialize()

dfparams['dfparams'].to_csv('cluster_param.csv', index=False)

ModuleNotFoundError: No module named 'Names'

<Figure size 585.25x413.875 with 0 Axes>

## Spatial Plot and F275W-F336W vs F275W Color-Magnitude Diagram of cluster(s)

In [ ]:
clusterlist = {
    'ngc2808': dfclusters['ngc2808'],
    'ngc0104': dfclusters['ngc0104']
}

import spatialplot
importlib.reload(spatialplot)
import plotcmd
importlib.reload(plotcmd)

# spatialplot.fewclusters(clusterlist, dpi=100, legend=False, afterhowmany=3) # 'afterhowmany' is the after how many rows shall the code pick the next row to print - aim is to reduce computational time. 
# spatialplot.allclusters(dfclusters, dpi=100, legend=True, afterhowmany=10)

# plotcmd.fewclusters(func_analysis.cleandata, dfparams, names['dictfilters'], clusterlist, dpi=100, pmholder=[-12], legend=True)
# plotcmd.allclusters(func_analysis.cleandata, dfparams, names['dictfilters'], dfclusters, dpi=100, pmholder=[-12], legend=True)


## Pre-analysis checks on models

In [ ]:
clustername = 'ngc5272'
wantabund=False 

racluster = round(dfparams['dfparams'].loc[dfparams['dfparams']['ID'] == clustername, "RA"].values[0], 4)
deccluster = round(dfparams['dfparams'].loc[dfparams['dfparams']['ID'] == clustername, "DEC"].values[0], 4)
metallicity = round(dfparams['dfparams'].loc[dfparams['dfparams']['ID'] == clustername, "[Fe/H]"].values[0], 3)
reddening = round(dfparams['dfparams'].loc[dfparams['dfparams']['ID'] == clustername, "E(B-V)"].values[0], 3)
DM = dfparams['dfparams'].loc[dfparams['dfparams']['ID'] == clustername, "(m-M)V"].values[0]
aDM = round(DM - 3.1*reddening, 2)
d = round(10 * (10 ** ((aDM) / 5)) - (3.1 * (reddening)), 2)

print("Parameters of " + clustername + " from Harris Catalogue:")
print("RA and DEC: " + str(racluster) + " " + str(deccluster))
print("Metallicity: " + str(metallicity)) # use this metallicity value to dowload models
print("Reddening: " + str(reddening))
print("Absolute Distance Modulus: " + str(DM))
print("Apparent Distance Modulus: " + str(aDM))
print("Distance in pc: " + str(d))

In [ ]:
ageiso = 12000
c_masset = 80
a_masset1 = 70
a_masset2 = 60
wdmass = '054'

# The following models are input for the msto.py and wd.py directly. 
# Please check before calling the MSTO and WD routines. 
# One can skip abundantet1 and abundantet2 by putting wantabund=False.
# 1 Isochrone and 1 ET is essential to run the MSTO routine. 
# 1 WD Model is essential to run the WD routine.

# the apparent distance modulus and reddening value used can be adjusted. 
# I have done this below as adding some value to aDM and reddening.
# One can also artifically redden or de-redden the models. This is done by adding or subtracting some value from 'F336W' filter.

# increasing aDM shifts the model to lower magnitudes
# increasing reddening shifts the model to lower magnitudes and reddens the model

c_isohold = models['iso162'][str(ageiso)]
canonicaliso_dict = func_analysis.corrections(aDM, reddening+0.01, {'main': c_isohold[0:600]})
canonicaliso = canonicaliso_dict['main'] 
canonicaliso['F336W'] -= 0.06
# print(canonicaliso)

c_ethold = models['et162_c'][str(c_masset)]
canonicalet_dict = func_analysis.corrections(aDM, reddening+0.01, {'main': c_ethold[0:300]})
canonicalet = canonicalet_dict['main']
canonicalet['F336W'] -= 0.06

a_et1hold = models['et131_a1'][str(a_masset1)]
abundantet1_dict = func_analysis.corrections(aDM-0.15, reddening-0.025, {'main': a_et1hold[100:350]})
abundantet1 = abundantet1_dict['main']
abundantet1['F336W'] -= 0.1

a_et2hold = models['et131_a2'][str(a_masset2)]
abundantet2_dict = func_analysis.corrections(aDM-0.11, reddening, {'main': a_et2hold[100:350]})
abundantet2 = abundantet2_dict['main']
abundantet2['F336W'] -= 0.17

wd_hold = models['wdCOHZ0420'][wdmass]
wdcooling_dict = func_analysis.corrections(aDM, reddening, {'main': wd_hold})
wdcooling = wdcooling_dict['main'] 
wdcooling['F336W'] -= 0.0  

import matplotlib.pyplot as plt

dictholdmodels_ms = {'He-canonical Iso': canonicaliso, 
                     'He-canonical ET': canonicalet, 
                     'He-abundant ET 1': abundantet1, 
                     'He-abundant ET 2': abundantet2, 
                     'WD Cooling Track':wdcooling}

plt.tight_layout()
plt.rcParams['figure.dpi']=100
fig=plt.figure(figsize=(10,10))

plt.scatter(dfclusters[clustername]['F275W'] - dfclusters[clustername]['F336W'], dfclusters[clustername]['F275W'], 
            marker = 'o', s=2, alpha=1, color='black', edgecolors='none')


for i, (label, modelhere) in enumerate(dictholdmodels_ms.items()):
    if not wantabund and i in (2, 3):
        continue
    plt.plot(modelhere['F275W'] - modelhere['F336W'],
             modelhere['F275W'],
             color=names['colors'][i], lw=1, linestyle='-', markersize=0.1,
             marker='o', label=label)

plt.ylim(13, 24.5)
plt.xlim(-2, 4)
plt.gca().invert_yaxis()
plt.legend()

plt.show()     

## Major Routines

In [ ]:
import msto
importlib.reload(msto)

Completeness=False

msto_ret = msto.initialize(func_analysis.cleandata(dfclusters[clustername], 80),
                ageiso, c_masset, str(a_masset1), str(a_masset2),
                canonicaliso, canonicalet, abundantet1, abundantet2,
                clustername, wantabund, Completeness,
                delalpholder = [0.0040], # this can be loaded with more values.
                g=0, dpi=200,
                pm = 80, bluelimit = 0.45, redlimit = 0.7, sigmacut = 1.5, 
                wantcsv=False, 
                wantlegend=False # printing legend won't be required since everything is being printed as output in text.
                )

SPUpperMS = msto_ret['concatnewupper']
SPLowerMS = msto_ret['concatnewlower']

MSSC_c = len(msto_ret['MSSC_c'])
if Completeness:
    MSSC_c_comp = len(msto_ret['MSSC_c_comp'])
MSCT_c = abs(msto_ret['MSCT_c'])

if wantabund:
    MSCT_a1 = msto_ret['MSCT_a1']
    MSCT_a2 = msto_ret['MSCT_a2']

In [ ]:
import wd
importlib.reload(wd)

wd_ret = wd.initialize(func_analysis.cleandata(dfclusters[clustername], -12),
                wdcooling, clustername,
                max_allowed_mag=24, min_allowed_mag=20.2, # these numbers come from AST
                dpi=100,
                binsize=0.2, # can be increased or decreased as per choice.
                division=22.5, # can be increased or decreased as per cluster.
                min_mag=18, max_mag=26, min_col=-1.5, max2=0.2,
                wdcolmin = -1, wdcolmax=-0.35,
                wdcolminbri = -0.25, wdcolmaxbri=-1,
                briwdstart = 18,
                wantlegend=False, Completeness=False
              )

SPUpperWD = wd_ret['bin1']
SPLowerWD = wd_ret['bin2'] 
SPBrightWD = wd_ret['dfwdcalbright']

WDCTup = wd_ret['WDCTup']
WDSCup = wd_ret['sum_values_upper_ncorr']
WDSCup_corr = wd_ret['sum_values_upper_corr']

WDCTlo = wd_ret['WDCTlo']
WDSClo = wd_ret['sum_values_lower_ncorr'] + WDSCup
WDSClo_corr = wd_ret['sum_values_lower_corr'] + WDSCup_corr

## WD/MSTO Ratio Calulcation

In [ ]:
import ratios
importlib.reload(ratios)

rationumholder = {0: [WDSCup, MSSC_c, WDCTup, MSCT_c],
                  1: [WDSClo, MSSC_c, WDCTlo, MSCT_c]
                  }

ratios.initialize(rationumholder)

In [ ]:
# import pandas as pd
# SPhb = pd.DataFrame() # SPhb under work for generalization.

In [ ]:
# import stelpha_splplt_raddist
# importlib.reload(stelpha_splplt_raddist)

# stelpha_splplt_raddist.initialize(func_analysis.cleandata(dfclusters[clustername], -12), 
#                                   func_analysis.cleandata(dfclusters[clustername], 80), 
#                                   racluster, deccluster, 
#                                   SPUpperMS, SPLowerMS,
#                                   SPUpperWD, SPLowerWD, SPBrightWD,
#                                   SPhb,
#                                   singlecolor=False, 
#                                   hb=False,
#                                   dpi=100)

In [ ]:
# # External Field of NGC2808 Photometric Data
# dfExtF2808 = func_input.InputFile('#      ', "/Users/lakshgupta/Downloads/Personal/NGC2808WDProject/Theoretical Models/NGC2808PhotometricData/ExtF2808/NGC2808.METH3.CAT") # provided by Domenico Nardiello
# dfspatialEXT = dfExtF2808
# index_list_df_spatial_ext = dfspatialEXT.index.tolist()